# Capítulo 10 · Delineamentos experimentais e quase-experimentais em saúde: caderno complementar
*Ciência de Dados e Inteligência Artificial em Saúde, Volume 1*

> **Este caderno acompanha o capítulo do livro; não o reproduz.** Definições, deduções e explicações estão no livro. Aqui você executa o código do capítulo sobre o caso e confere os números. Todos os dados são **sintéticos**; nenhum paciente real aparece.

**O caso.** Um ambulatório de hipertensão e seus 18 consultórios de referência decidem se oferecem dois anti-hipertensivos em uma pílula única em vez de comprimidos separados; o desfecho é a pressão arterial não controlada em 12 semanas. Três arquivos representam três avaliações alternativas dessa mesma decisão (não são seguimento vinculado dos mesmos pacientes): `data/trial.csv` (ensaio randomizado individual, 1.200 adultos), `data/cluster.csv` (ensaio randomizado por consultórios, 1.800 adultos nos consultórios P01-P18) e `data/monthly.csv` (registro de 24 meses, consultórios M01-M18).

A **Parte A** executa o código do capítulo seção por seção. A **Parte B** traz o código e o resultado dos exercícios do capítulo que exigem código (os enunciados estão no livro). Execute as células em ordem. As saídas do código usam ponto decimal.

## Parte A. O caso

### A comparação e o risco do desfecho

In [1]:
import pandas as pd
trial = pd.read_csv("data/trial.csv")
events = trial["uncontrolled_12w"].eq("yes").sum()
print(f"Patients: {len(trial)}; uncontrolled: {events}")
print(f"Overall risk: {events / len(trial):.4f}")

Patients: 1200; uncontrolled: 424
Overall risk: 0.3533


### Randomização e equilíbrio basal

In [2]:
import pandas as pd
trial = pd.read_csv("data/trial.csv")
for arm, group in trial.groupby("allocated"):
    high = group["baseline_sbp"].ge(160).sum()
    print(f"{arm}: {high}/{len(group)} = {100*high/len(group):.1f}%")

single pill: 205/596 = 34.4%
two tablets: 223/604 = 36.9%


### Intenção de tratar e incerteza

In [3]:
import pandas as pd
trial = pd.read_csv("data/trial.csv")
counts = pd.crosstab(trial["allocated"], trial["uncontrolled_12w"])
risks = counts["yes"] / counts.sum(axis=1)
p, q = risks.loc["single pill"], risks.loc["two tablets"]
for arm in counts.index:
    print(f"{arm}: {counts.loc[arm, 'yes']}/{counts.loc[arm].sum()}")
print(f"Risk difference: {100*(p-q):.1f} percentage points")
print(f"Risk ratio: {p/q:.2f}")

single pill: 184/596
two tablets: 240/604
Risk difference: -8.9 percentage points
Risk ratio: 0.78


In [4]:
import pandas as pd
from math import sqrt, log, exp
from statistics import NormalDist
trial = pd.read_csv("data/trial.csv")
x = pd.crosstab(trial["allocated"], trial["uncontrolled_12w"])
a, b = x.loc["single pill", "yes"], x.loc["two tablets", "yes"]
n, m = x.loc["single pill"].sum(), x.loc["two tablets"].sum()
p, q = a/n, b/m
z = NormalDist().inv_cdf(0.975)
d = 100*(p-q)
se = 100*sqrt(p*(1-p)/n + q*(1-q)/m)
sl = sqrt(1/a-1/n+1/b-1/m)
print(f"SE: {se:.4f}; normal multiplier: {z:.4f}")
print(f"Difference: {d:.1f}; 95% CI {d-z*se:.1f} to {d+z*se:.1f}")
print(f"Ratio: {p/q:.2f}; 95% CI "
      f"{exp(log(p/q)-z*sl):.2f} to {exp(log(p/q)+z*sl):.2f}")

SE: 2.7469; normal multiplier: 1.9600
Difference: -8.9; 95% CI -14.2 to -3.5
Ratio: 0.78; 95% CI 0.67 to 0.91


### Perguntas por protocolo e tratamento recebido

In [5]:
import pandas as pd
from math import sqrt, log, exp
from statistics import NormalDist
trial = pd.read_csv("data/trial.csv")
same = trial["allocated"].eq("single pill") == trial["received"].eq("yes")
x = pd.crosstab(trial.loc[same, "allocated"],
                trial.loc[same, "uncontrolled_12w"])
a, b = x.loc["single pill", "yes"], x.loc["two tablets", "yes"]
n, m = x.loc["single pill"].sum(), x.loc["two tablets"].sum()
rr = (a/n)/(b/m)
se = sqrt(1/a-1/n+1/b-1/m)
z = NormalDist().inv_cdf(0.975)
print(f"Retained: {n+m}; events/total: {a}/{n}, {b}/{m}")
print(f"Restricted ratio: {rr:.2f}; 95% CI "
      f"{exp(log(rr)-z*se):.2f} to {exp(log(rr)+z*se):.2f}")

Retained: 1122; events/total: 162/544, 230/578
Restricted ratio: 0.75; 95% CI 0.64 to 0.88


### Ensaios randomizados por conglomerados

In [6]:
import pandas as pd
cluster = pd.read_csv("data/cluster.csv")
for arm, group in cluster.groupby("arm"):
    a = group["uncontrolled_12w"].eq("yes").sum()
    k = group["practice"].nunique()
    print(f"{arm}: {k} practices; {a}/{len(group)}")
x = pd.crosstab(cluster["arm"], cluster["uncontrolled_12w"])
r = x["yes"] / x.sum(axis=1)
print(f"Pooled ratio: {r.loc['single pill first']/r.loc['usual prescribing']:.2f}")

single pill first: 9 practices; 271/876
usual prescribing: 9 practices; 395/924
Pooled ratio: 0.72


In [7]:
import pandas as pd
from math import sqrt
from scipy.stats import t
cluster = pd.read_csv("data/cluster.csv")
cluster["y"] = cluster["uncontrolled_12w"].eq("yes")
r = cluster.groupby(["arm", "practice"])["y"].mean()
p, q = r.loc["single pill first"], r.loc["usual prescribing"]
u, v = p.var()/len(p), q.var()/len(q)
df = (u+v)**2 / (u*u/(len(p)-1) + v*v/(len(q)-1))
d, se = 100*(p.mean()-q.mean()), 100*sqrt(u+v)
cut = t.ppf(0.975, df)
print(f"Mean practice risks: {p.mean():.4f}, {q.mean():.4f}")
print(f"SE: {se:.4f}; df: {df:.2f}; t cutoff: {cut:.4f}")
print(f"Difference: {d:.1f}; 95% CI {d-cut*se:.1f} to {d+cut*se:.1f}")

Mean practice risks: 0.3092, 0.4281
SE: 1.7601; df: 15.74; t cutoff: 2.1228
Difference: -11.9; 95% CI -15.6 to -8.2


### Comparações antes e depois e tendências seculares

In [8]:
import pandas as pd
monthly = pd.read_csv("data/monthly.csv")
p18 = monthly[monthly["practice"].eq("M18")]
risks = []
for name, rows in [("before", p18[p18["month"] < 7]),
                   ("after", p18[p18["month"] >= 7])]:
    a, n = rows["uncontrolled"].sum(), rows["patients"].sum()
    risks.append(a/n)
    print(f"{name}: {a}/{n} = {100*a/n:.1f}%")
print(f"After/before: {risks[1]/risks[0]:.2f}")

before: 298/586 = 50.9%
after: 533/1693 = 31.5%
After/before: 0.62


### Comparações controladas e diferença em diferenças

In [9]:
import pandas as pd
monthly = pd.read_csv("data/monthly.csv")
def risk_in(rows):
    return rows["uncontrolled"].sum() / rows["patients"].sum()
p = monthly[monthly["practice"].eq("M18")]
c = monthly[monthly["practice"].isin(["M01", "M02", "M03"])]
a0, a1 = risk_in(p[p["month"] < 7]), risk_in(p[p["month"] >= 7])
b0, b1 = risk_in(c[c["month"] < 7]), risk_in(c[c["month"] >= 7])
print(f"M18 risks: {100*a0:.1f}%, {100*a1:.1f}%")
print(f"Comparison risks: {100*b0:.1f}%, {100*b1:.1f}%")
print(f"Changes: {100*(a1-a0):.1f}, {100*(b1-b0):.1f} points")
print(f"Difference in differences: {100*((a1-a0)-(b1-b0)):.1f} points")
print(f"Assumed untreated endpoint: {100*(a0+b1-b0):.1f}%")
print(f"Ratio of ratios: {(a1/a0)/(b1/b0):.2f}")

M18 risks: 50.9%, 31.5%
Comparison risks: 43.8%, 38.1%
Changes: -19.4, -5.7 points
Difference in differences: -13.7 points
Assumed untreated endpoint: 45.2%
Ratio of ratios: 0.71


### Série temporal interrompida

In [10]:
import pandas as pd
monthly = pd.read_csv("data/monthly.csv")
p18 = monthly[monthly["practice"].eq("M18")].sort_values("month")
first = p18.iloc[0]
print(f"Before months: {(p18['month'] < 7).sum()}")
print(f"After months: {(p18['month'] >= 7).sum()}")
print(f"First risk: {first['uncontrolled']}/{first['patients']}"
      f" = {100*first['uncontrolled']/first['patients']:.1f}%")

Before months: 6
After months: 18
First risk: 65/109 = 59.6%


## Parte B. Soluções dos exercícios

Os exercícios sem código (respostas conceituais) são respondidos no livro.

### Exercício 1. Equilíbrio basal

In [11]:
import pandas as pd
trial = pd.read_csv("data/trial.csv")
for arm, group in trial.groupby("allocated"):
    high = group["baseline_sbp"].ge(160).sum()
    print(f"{arm}: n={len(group)}, age={group['age_years'].mean():.1f}, "
          f"high={high}, share={100*high/len(group):.1f}%")

single pill: n=596, age=61.7, high=205, share=34.4%
two tablets: n=604, age=61.2, high=223, share=36.9%


### Exercício 3. Perguntas sobre alocação e adesão

In [12]:
import pandas as pd
from math import sqrt, log, exp
from statistics import NormalDist
trial = pd.read_csv("data/trial.csv")
x = pd.crosstab(trial["allocated"], trial["uncontrolled_12w"])
a, b = x.loc["single pill", "yes"], x.loc["two tablets", "yes"]
n, m = x.loc["single pill"].sum(), x.loc["two tablets"].sum()
p, q = a/n, b/m
z = NormalDist().inv_cdf(0.975)
d = 100*(p-q)
se = 100*sqrt(p*(1-p)/n+q*(1-q)/m)
sl = sqrt(1/a-1/n+1/b-1/m)
print(f"Difference: {d:.1f}; 95% CI {d-z*se:.1f} to {d+z*se:.1f}")
print(f"Ratio: {p/q:.2f}; 95% CI "
      f"{exp(log(p/q)-z*sl):.2f} to {exp(log(p/q)+z*sl):.2f}")

Difference: -8.9; 95% CI -14.2 to -3.5
Ratio: 0.78; 95% CI 0.67 to 0.91


### Exercício 7. Pacientes dentro dos consultórios alocados

In [13]:
import pandas as pd
cluster = pd.read_csv("data/cluster.csv")
cluster["y"] = cluster["uncontrolled_12w"].eq("yes")
by_practice = cluster.groupby(["practice", "arm"])["y"].agg(
    events="sum", patients="size", risk="mean")
for row in by_practice.reset_index().itertuples(index=False):
    print(f"{row.practice}: {row.events}/{row.patients} = {100*row.risk:.1f}%")

P01: 38/109 = 34.9%
P02: 47/112 = 42.0%
P03: 43/101 = 42.6%
P04: 37/95 = 38.9%
P05: 48/115 = 41.7%
P06: 24/86 = 27.9%
P07: 27/105 = 25.7%
P08: 48/117 = 41.0%
P09: 29/105 = 27.6%
P10: 36/89 = 40.4%
P11: 50/107 = 46.7%
P12: 34/93 = 36.6%
P13: 27/86 = 31.4%
P14: 31/101 = 30.7%
P15: 39/120 = 32.5%
P16: 47/91 = 51.6%
P17: 22/71 = 31.0%
P18: 39/97 = 40.2%


In [14]:
import pandas as pd
from math import sqrt
from scipy.stats import t
cluster = pd.read_csv("data/cluster.csv")
cluster["y"] = cluster["uncontrolled_12w"].eq("yes")
r = cluster.groupby(["arm", "practice"])["y"].mean()
p, q = r.loc["single pill first"], r.loc["usual prescribing"]
u, v = p.var()/len(p), q.var()/len(q)
df = (u+v)**2/(u*u/(len(p)-1)+v*v/(len(q)-1))
d, se = 100*(p.mean()-q.mean()), 100*sqrt(u+v)
cut = t.ppf(0.975, df)
print(f"Difference: {d:.1f}; SE: {se:.4f}; df: {df:.2f}")
print(f"95% CI: {d-cut*se:.1f} to {d+cut*se:.1f} percentage points")

Difference: -11.9; SE: 1.7601; df: 15.74
95% CI: -15.6 to -8.2 percentage points


### Exercício 9. A mudança no registro como um todo

In [15]:
import pandas as pd
monthly = pd.read_csv("data/monthly.csv")
for name, rows in [("months 1-12", monthly[monthly["month"] <= 12]),
                   ("months 13-24", monthly[monthly["month"] >= 13])]:
    n = rows["patients"].sum()
    a, s = rows["uncontrolled"].sum(), rows["single_pill"].sum()
    print(f"{name}: uncontrolled {a}/{n} = {100*a/n:.1f}%; "
          f"uptake {s}/{n} = {100*s/n:.1f}%")

months 1-12: uncontrolled 9497/20414 = 46.5%; uptake 426/20414 = 2.1%
months 13-24: uncontrolled 7763/20595 = 37.7%; uptake 1806/20595 = 8.8%


### Exercício 10. M09 e sua comparação simultânea

In [16]:
import pandas as pd
monthly = pd.read_csv("data/monthly.csv")
def risk_in(rows):
    return rows["uncontrolled"].sum()/rows["patients"].sum()
p = monthly[monthly["practice"].eq("M09")]
c = monthly[monthly["practice"].isin(["M01", "M02", "M03"])]
a0, a1 = risk_in(p[p["month"] < 13]), risk_in(p[p["month"] >= 13])
b0, b1 = risk_in(c[c["month"] < 13]), risk_in(c[c["month"] >= 13])
after = p[p["month"] >= 13]
uptake = after["single_pill"].sum()/after["patients"].sum()
print(f"M09: {100*a0:.1f}%, {100*a1:.1f}%; ratio {a1/a0:.2f}")
print(f"Comparison: {100*b0:.1f}%, {100*b1:.1f}%; ratio {b1/b0:.2f}")
print(f"Difference in differences: {100*((a1-a0)-(b1-b0)):.1f} points")
print(f"After uptake: {100*uptake:.1f}%")

M09: 46.9%, 29.9%; ratio 0.64
Comparison: 42.3%, 36.7%; ratio 0.87
Difference in differences: -11.4 points
After uptake: 77.1%


---
*Dados sintéticos (semente 10, `generate_data.py`). Apenas hipóteses didáticas; nada aqui é recomendação clínica nem estimativa para um serviço real.*